# model_xgboost.py

XGBoost for Heart Disease / Cardiovascular Risk prediction.

Steps:
  1. Load preprocessed data
  2. Define model architecture
  3. Loss: softmax cross-entropy (multi:softmax / softprob)
  4. Optimiser: Adam-like (XGBoost internal) with learning_rate
  5. Train for N boosting rounds (epochs) with early stopping
  6. Evaluate
  7. Fine-tune with Optuna / RandomizedSearchCV
  8. Save model
  9. Predict on new data

In [8]:
pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.1/101.7 MB 1.3 MB/s eta 0:01:17
   ---------------------------------------- 0.3/101.7 MB 3.3 MB/s eta 0:00:32
   ---------------------------------------- 0.9/101.7 MB 6.4 MB/s eta 0:00:16
    --------------------------------------- 1.5/101.7 MB 7.8 MB/s eta 0:00:13
   - -------------------------------------- 2.9/101.7 MB 12.4 MB/s eta 0:00:08
   - -------------------------------------- 5.0/101.7 MB 16.7 MB/s eta 0:00:06
   -- ------------------------------------- 6.3/101.7 MB 19.1 MB/s eta 0:00:06
   -- ------------------------------------- 7.5/101.7 MB 20.1 MB/s eta 0:00:05
   --- ------------------------------------ 9.1/101.7 MB 20.7 MB/s eta 0:00:05
   ---- ----------------------------------- 11.2/101.7 MB 32.7 MB/s eta 0:00:03
   ----- ---------------------------------- 12.7/101.7 MB 32.8 MB/s eta

In [9]:
import numpy as np
import pandas as pd
import joblib, os
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, accuracy_score, ConfusionMatrixDisplay)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────────
PRE  = "preprocessed"
MDIR = "models/xgb"; os.makedirs(MDIR, exist_ok=True)
PDIR = "plots/xgb";  os.makedirs(PDIR, exist_ok=True)
LABEL = {0: 'Low', 1: 'Moderate', 2: 'High'}

## 1. Load preprocessed splits

In [11]:
X_train = pd.read_csv(f"{PRE}/X_train.csv")
X_val   = pd.read_csv(f"{PRE}/X_val.csv")
X_test  = pd.read_csv(f"{PRE}/X_test.csv")
y_train = pd.read_csv(f"{PRE}/y_train.csv").squeeze()
y_val   = pd.read_csv(f"{PRE}/y_val.csv").squeeze()
y_test  = pd.read_csv(f"{PRE}/y_test.csv").squeeze()
class_weights = joblib.load(f"{PRE}/class_weights.pkl")
feature_names = joblib.load(f"{PRE}/feature_names.pkl")
print(f"Train {X_train.shape} | Val {X_val.shape} | Test {X_test.shape}")

X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])

Train (51229, 30) | Val (10978, 30) | Test (10978, 30)


## 2. Define model architecture

In [40]:
# Build sample-weights array from class_weights
sample_weights = y_train.map(class_weights).values

NUM_CLASSES    = 3
N_ESTIMATORS   = 1500        # max rounds (epochs)
EARLY_STOP     = 50         # 5. early stopping patience
LEARNING_RATE  = 0.03       # 4. learning rate

# 2. Architecture: gradient boosted trees with dart/gbtree
xgb_base = xgb.XGBClassifier(
    objective        = 'multi:softprob',
    eval_metric      = ['mlogloss', 'merror'],
    num_class        = NUM_CLASSES,
    n_estimators     = N_ESTIMATORS,
    learning_rate    = LEARNING_RATE,
    max_depth        = 6,
    min_child_weight = 3,
    gamma            = 0.1,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.05,
    reg_lambda       = 1.0,
    tree_method      = 'hist',
    # early_stopping_rounds removed from here
    random_state     = 42,
)

## 3+4. Loss = multi:softprob cross-entropy Optimiser = gradient boosting (Newton-Raphson / Adam-like)

## 5. Train with early stopping (eval_set acts as validation)

In [50]:
print("\n── Training XGBoost ──")
xgb_base.fit(
    X_train, y_train,
    sample_weight = sample_weights,
    eval_set      = [(X_train, y_train), (X_val, y_val)],
    verbose       = 50,
)

# Manually find best round from val loss curve
results    = xgb_base.evals_result()
val_loss   = results['validation_1']['mlogloss']
best_round = int(np.argmin(val_loss))
print(f"Best round: {best_round}  (val mlogloss: {val_loss[best_round]:.5f})")


── Training XGBoost ──
[0]	validation_0-mlogloss:1.06893	validation_0-merror:0.06073	validation_1-mlogloss:1.06914	validation_1-merror:0.06759
[50]	validation_0-mlogloss:0.39148	validation_0-merror:0.02184	validation_1-mlogloss:0.39845	validation_1-merror:0.02906
[100]	validation_0-mlogloss:0.23719	validation_0-merror:0.02145	validation_1-mlogloss:0.24894	validation_1-merror:0.02897
[150]	validation_0-mlogloss:0.18029	validation_0-merror:0.01892	validation_1-mlogloss:0.19619	validation_1-merror:0.02760
[200]	validation_0-mlogloss:0.15044	validation_0-merror:0.01665	validation_1-mlogloss:0.16971	validation_1-merror:0.02633
[250]	validation_0-mlogloss:0.12984	validation_0-merror:0.01382	validation_1-mlogloss:0.15193	validation_1-merror:0.02423
[300]	validation_0-mlogloss:0.11226	validation_0-merror:0.01175	validation_1-mlogloss:0.13700	validation_1-merror:0.02241
[350]	validation_0-mlogloss:0.09735	validation_0-merror:0.00945	validation_1-mlogloss:0.12423	validation_1-merror:0.02031
[40

## 6. Evaluate on Test Set

In [52]:
print("\n── Evaluation on Test Set ──")
y_pred  = xgb_base.predict(X_test)
y_proba = xgb_base.predict_proba(X_test)
print(classification_report(y_test, y_pred, target_names=[LABEL[i] for i in sorted(LABEL)]))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_proba, multi_class='ovr'))


── Evaluation on Test Set ──
              precision    recall  f1-score   support

         Low       0.99      1.00      1.00      7669
    Moderate       0.98      0.99      0.99      3173
        High       0.70      0.27      0.39       136

    accuracy                           0.99     10978
   macro avg       0.89      0.75      0.79     10978
weighted avg       0.99      0.99      0.99     10978

Accuracy : 0.9881581344507196
ROC-AUC  : 0.9246284848654099


## 7. Fine-Tune with RandomizedSearchCV

In [54]:
print("\n── Fine-Tuning (RandomizedSearchCV) ──")
param_dist = {
    'n_estimators'    : [300, 400, 500, 600],
    'learning_rate'   : [0.01, 0.03, 0.05, 0.1],
    'max_depth'       : [4, 5, 6, 7, 8],
    'min_child_weight': [1, 3, 5],
    'subsample'       : [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'gamma'           : [0, 0.1, 0.2, 0.3],
    'reg_alpha'       : [0, 0.05, 0.1],
    'reg_lambda'      : [0.5, 1.0, 1.5],
}
xgb_search = xgb.XGBClassifier(
    objective='multi:softprob', num_class=NUM_CLASSES,
    tree_method='hist', random_state=42, use_label_encoder=False,
    eval_metric='mlogloss'
)
rscv = RandomizedSearchCV(
    xgb_search, param_dist,
    n_iter=25, cv=StratifiedKFold(3),
    scoring='roc_auc_ovr', n_jobs=-1, verbose=1, random_state=42
)
rscv.fit(X_trainval, y_trainval)
best_xgb = rscv.best_estimator_
print("Best params:", rscv.best_params_)

y_pred_ft  = best_xgb.predict(X_test)
y_proba_ft = best_xgb.predict_proba(X_test)
print("Fine-tuned Accuracy :", accuracy_score(y_test, y_pred_ft))
print("Fine-tuned ROC-AUC  :", roc_auc_score(y_test, y_proba_ft, multi_class='ovr'))
print(classification_report(y_test, y_pred_ft, target_names=[LABEL[i] for i in sorted(LABEL)]))


── Fine-Tuning (RandomizedSearchCV) ──
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Best params: {'subsample': 0.9, 'reg_lambda': 0.5, 'reg_alpha': 0.1, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.03, 'gamma': 0.3, 'colsample_bytree': 0.7}
Fine-tuned Accuracy : 0.9893423210056477
Fine-tuned ROC-AUC  : 0.9429757352187647
              precision    recall  f1-score   support

         Low       0.99      1.00      1.00      7669
    Moderate       0.98      0.99      0.99      3173
        High       0.97      0.26      0.42       136

    accuracy                           0.99     10978
   macro avg       0.98      0.75      0.80     10978
weighted avg       0.99      0.99      0.99     10978



## 8. Save models

In [56]:
xgb_base.save_model(f"{MDIR}/xgb_base.json")
best_xgb.save_model(f"{MDIR}/xgb_finetuned.json")
print(f"✅ Models saved to ./{MDIR}/")

✅ Models saved to ./models/xgb/


## 9. Predict on new data

In [61]:
def predict_new(raw_df: pd.DataFrame, model_path: str = f"{MDIR}/xgb_finetuned.json"):
    """
    Accepts a pre-processed DataFrame (post preprocessing.py).
    Returns (predicted_labels, probabilities).
    """
    scaler        = joblib.load(f"{PRE}/scaler.pkl")
    feature_names = joblib.load(f"{PRE}/feature_names.pkl")
    model = xgb.XGBClassifier()
    model.load_model(model_path)

    raw_df = raw_df.reindex(columns=feature_names, fill_value=0)

    numerical_cols = [
        'Height_(cm)', 'Weight_(kg)', 'BMI', 'Alcohol_Consumption',
        'Fruit_Consumption', 'Green_Vegetables_Consumption', 'FriedPotato_Consumption',
        'Checkup_Frequency', 'Lifestyle_Score', 'Healthy_Diet_Score',
        'Smoking_Alcohol', 'Checkup_Exercise', 'Height_to_Weight',
        'Fruit_Vegetables', 'HealthyDiet_Lifestyle', 'Alcohol_FriedPotato',
        'Risk_Composite', 'Alcohol_per_Weight'
    ]
    num_present = [c for c in numerical_cols if c in raw_df.columns]
    raw_df[num_present] = scaler.transform(raw_df[num_present])

    preds  = model.predict(raw_df)
    probas = model.predict_proba(raw_df)
    return [LABEL[p] for p in preds], probas

# Demo
demo_preds = best_xgb.predict(X_test.iloc[:5].values)
print("\nDemo predictions:", [LABEL[p] for p in demo_preds],
      "| Actual:", list(y_test.iloc[:5].map(LABEL)))


Demo predictions: ['Low', 'Moderate', 'Low', 'Low', 'Low'] | Actual: ['Low', 'Moderate', 'Low', 'Low', 'Low']
